In [2]:
import pandas as pd

df = pd.read_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\applemint\applemint.csv').drop('Weight', axis=1)
df.columns

Index(['Product', 'Handle', 'Price', 'SKU', 'Availability', 'Variant Title',
       'Variant ID', 'Inventory Management', 'Product URL', 'Scraped at',
       'Description'],
      dtype='object')

In [3]:
df = df.rename({'Variant Title': 'Variant'}, axis=1)
df

,Product,Handle,Price,SKU,Availability,Variant,Variant ID,Inventory Management,Product URL,Scraped at,Description
0,Adventure Incomplete Washed Tee For Adults,adventure-incomplete-tee-for-adults,519.0,NaN,False,Medium / T-shirt Only,46473465069817,shopify,https://applemintandcocoa.com//products/advent...,2026/02/22 12:53:26 PM,Size & Fit:Oversized FitFabric:100% Cotton Dis...
1,Adventure Incomplete Washed Tee For Adults,adventure-incomplete-tee-for-adults,519.0,NaN,False,Large / T-shirt Only,46473465102585,shopify,https://applemintandcocoa.com//products/advent...,2026/02/22 12:53:26 PM,Size & Fit:Oversized FitFabric:100% Cotton Dis...
2,Adventure Incomplete Washed Tee For Adults,adventure-incomplete-tee-for-adults,519.0,NaN,True,X-Large / T-shirt Only,46473465135353,shopify,https://applemintandcocoa.com//products/advent...,2026/02/22 12:53:26 PM,Size & Fit:Oversized FitFabric:100% Cotton Dis...
3,Adventure Incomplete Washed Tee For Adults,adventure-incomplete-tee-for-adults,519.0,NaN,True,XX-Large / T-shirt Only,46473465168121,shopify,https://applemintandcocoa.com//products/advent...,2026/02/22 12:53:26 PM,Size & Fit:Oversized FitFabric:100% Cotton Dis...
4,Aiming Higher in Olive Sports Tee,aiming-higher-in-olive-tee,322.0,NaN,True,4-5,46516044366073,shopify,https://applemintandcocoa.com//products/aiming...,2026/02/22 12:53:27 PM,"Introducing -PULSE- Comfortable, breathable sp..."
...,...,...,...,...,...,...,...,...,...,...,...
4040,Twirl in Cashmere Girls Sports Tee,twirl-in-powder-pink-girls-tee,322.0,NaN,True,6-7,46461443375353,shopify,https://applemintandcocoa.com//products/twirl-...,2026/02/22 12:59:37 PM,"Introducing -PULSE- Comfortable, breathable sp..."
4041,Twirl in Cashmere Girls Sports Tee,twirl-in-powder-pink-girls-tee,322.0,NaN,True,8-9,46461443408121,shopify,https://applemintandcocoa.com//products/twirl-...,2026/02/22 12:59:37 PM,"Introducing -PULSE- Comfortable, breathable sp..."
4042,Twirl in Cashmere Girls Sports Tee,twirl-in-powder-pink-girls-tee,322.0,NaN,False,10-11,46461443440889,shopify,https://applemintandcocoa.com//products/twirl-...,2026/02/22 12:59:37 PM,"Introducing -PULSE- Comfortable, breathable sp..."
4043,Twirl in Cashmere Girls Sports Tee,twirl-in-powder-pink-girls-tee,322.0,NaN,True,12-13,46461443473657,shopify,https://applemintandcocoa.com//products/twirl-...,2026/02/22 12:59:37 PM,"Introducing -PULSE- Comfortable, breathable sp..."


In [4]:
import pandas as pd

# 1. Clean the 'Description' column BEFORE merging to remove new lines
df['Description'] = df['Description'].str.replace(r'\n|\r', ' ', regex=True)

# 2. Updated aggregation rules to create clean strings instead of Python lists
# We use ', '.join() to make "White, Cream" instead of "['White', 'Cream']"
aggregation_rules = {
    # 'Title': lambda x: ', '.join(map(str, x.unique())),
    'Variant': lambda x: ', '.join(map(str, x.unique())), 
    'Variant ID': lambda x: ', '.join(map(str, x.unique())),
    'SKU': lambda x: ', '.join(map(str, x.unique())),
    # 'Size': lambda x: ', '.join(map(str, x.unique())),
    'Inventory Management': 'first',
    'Handle': 'first',
    'Availability': 'first',
    'Product URL': 'first',
    'Scraped at': 'first'
}

# Group by the "Core" identity
df_merged = df.groupby(['Product', 'Description', 'Price'], as_index=False).agg(aggregation_rules)

# 3. FIX: Create the Hollistic Description correctly
# Use .astype(str) instead of str() to handle it row-by-row
df_merged['Hollistic Description'] = (
    df_merged['Product'] + 
    ' - ' + 
    df_merged['Variant'].astype(str) + 
    ": " + 
    df_merged["Description"]
)

# Reorder columns
columns_order = [
    'Product', 'Variant', 'Price', 'Handle', 
    'Inventory Management', 'Variant ID', 'Availability', 'SKU', 
    'Product URL', 'Scraped at',
    'Description', 'Hollistic Description'
]
df_merged = df_merged[columns_order]

# Save to CSV (index=False prevents that extra "0, 1, 2" column at the start)
df_merged.to_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\applemint\applemint_compact.csv', index=False)
df_merged

,Product,Variant,Price,Handle,Inventory Management,Variant ID,Availability,SKU,Product URL,Scraped at,Description,Hollistic Description
0,Adventure Incomplete Washed Tee For Adults,"Medium / T-shirt Only, Large / T-shirt Only, X...",519.0,adventure-incomplete-tee-for-adults,shopify,"46473465069817, 46473465102585, 46473465135353...",False,nan,https://applemintandcocoa.com//products/advent...,2026/02/22 12:53:26 PM,Size & Fit:Oversized FitFabric:100% Cotton Dis...,Adventure Incomplete Washed Tee For Adults - M...
1,Adventure Incomplete Washed Tee For Kids,"2-3 Years / T-shirt Only, 3-4 Years / T-shirt ...",369.0,adventure-incomplete-washed-tee-for-kids,shopify,"46473468018937, 46473468051705, 46473468084473...",False,nan,https://applemintandcocoa.com//products/advent...,2026/02/22 12:53:37 PM,Size & Fit: Oversized Fit Fabric: 100% Cotton ...,Adventure Incomplete Washed Tee For Kids - 2-3...
2,Aiming Higher in Olive Sports Tee,"4-5, 6-7, 8-9, 10-11, 12-13, 14-15",322.0,aiming-higher-in-olive-tee,shopify,"46516044366073, 46461442031865, 46461442064633...",True,nan,https://applemintandcocoa.com//products/aiming...,2026/02/22 12:53:27 PM,"Introducing -PULSE- Comfortable, breathable sp...","Aiming Higher in Olive Sports Tee - 4-5, 6-7, ..."
3,Airborne Unicorn,"2-3, 4-5, 6-7, 8-9, 10-11, 12-13",299.0,airborne-unicorn,shopify,"42544661397753, 42544661430521, 42544661463289...",False,nan,https://applemintandcocoa.com//products/airbor...,2026/02/22 12:53:28 PM,Crafted in our soft cotton jersey. Perfect for...,"Airborne Unicorn - 2-3, 4-5, 6-7, 8-9, 10-11, ..."
4,Airforce Sports Tee,"4-5, 6-7, 8-9, 10-11, 12-13, 14-15",299.0,airforce-tee,shopify,"45170394169593, 45170394202361, 45170394235129...",True,nan,https://applemintandcocoa.com//products/airfor...,2026/02/22 12:53:36 PM,"Introducing -PULSE- Comfortable, breathable sp...","Airforce Sports Tee - 4-5, 6-7, 8-9, 10-11, 12..."
...,...,...,...,...,...,...,...,...,...,...,...,...
689,Unisex Fleece Cargo Double Pocket Jogger Pants...,"2-3, 3-4, 4-5, 6-7, 8-9, 10-11, 12-13",650.0,unisex-fleece-cargo-double-pocket-jogger-pants...,shopify,"45912880611577, 45912880644345, 45912880677113...",True,nan,https://applemintandcocoa.com//products/unisex...,2026/02/22 12:59:34 PM,Key Features: 100% Cotton Fleece brushed insid...,Unisex Fleece Cargo Double Pocket Jogger Pants...
690,Unisex Fleece Cargo Jogger Pants in Mustard,"12-18M, 2-3, 3-4, 4-5, 6-7, 8-9, 10-11, 12-13",650.0,unisex-fleece-cargo-jogger-pants-in-havan,shopify,"44068257366265, 44068256579833, 44068256612601...",True,nan,https://applemintandcocoa.com//products/unisex...,2026/02/22 12:59:27 PM,Key Features: 100% Cotton Fleece brushed insid...,Unisex Fleece Cargo Jogger Pants in Mustard - ...
691,White Cargo Girls Shorts,"2-3 Years / Shorts Only, 3-4 Years / Shorts On...",399.0,white-girls-shorts,shopify,"43732626309369, 43732626342137, 43732626374905...",False,nan,https://applemintandcocoa.com//products/white-...,2026/02/22 12:59:29 PM,Size & Fit: Regular fit. Runs true to size.Do ...,White Cargo Girls Shorts - 2-3 Years / Shorts ...
692,Winner in Black Sports Shorts,"6-7, 8-9, 10-11, 12-13, 14-15",370.0,winner-sports-shorts,shopify,"46461419290873, 46461419323641, 46461419356409...",True,nan,https://applemintandcocoa.com//products/winner...,2026/02/22 12:59:31 PM,"Introducing -PULSE- Comfortable, breathable sp...","Winner in Black Sports Shorts - 6-7, 8-9, 10-1..."


In [9]:
'''
Products Table
'''
df_products = df_merged
df_products = df_products.drop(columns=['Variant', 'Variant ID', 'Availability'])

# Split by comma and strip whitespace from each element
# 1. First, split the strings into lists (if you haven't already)
df_products['SKUs'] = df_products['SKU'].str.split(',')

# 2. Grab the first element of each list and strip whitespace
df_products['SKU'] = df_products['SKUs'].str[0].str.strip()
# 1. Clean up the decimals and whitespace within the lists first
# This removes '.0' and extra spaces from every item in every list
df_products['SKUs'] = df_products['SKUs'].apply(lambda x: [i.replace('.0', '').strip() for i in x])

# 2. Join the list elements into a single string separated by a space
df_products['SKUs'] = df_products['SKUs'].str.join(' ')

df_products.to_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\applemint\applemint_products.csv', index = False)

In [ ]:
'''
Variants Table
'''

df_variants = df
df_variants = df_variants.drop(columns=['Product', "Product URL", 'Description'])
cols_to_move = ['Variant ID', 'Handle', 'Variant', 'Price', 'SKU', 'Scraped at']

# Get the list of all other columns that aren't in the "move" list
remaining_cols = [c for c in df_variants.columns if c not in cols_to_move]

# Reorder with move-list first, then the rest
df_variants = df[cols_to_move + remaining_cols]
df_variants.to_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\applemint\applemint_variants.csv', index=False)
print(r'Successfully printed to applemint\applemint_variants.csv')
df_variants

Successfully printed to legacy_app\output\heba\heba_variants.csv


,Variant ID,Handle,Variant,Price,SKU,Scraped at,Availability,Inventory Management
0,46473465069817,adventure-incomplete-tee-for-adults,Medium / T-shirt Only,519.0,NaN,2026/02/22 12:53:26 PM,False,shopify
1,46473465102585,adventure-incomplete-tee-for-adults,Large / T-shirt Only,519.0,NaN,2026/02/22 12:53:26 PM,False,shopify
2,46473465135353,adventure-incomplete-tee-for-adults,X-Large / T-shirt Only,519.0,NaN,2026/02/22 12:53:26 PM,True,shopify
3,46473465168121,adventure-incomplete-tee-for-adults,XX-Large / T-shirt Only,519.0,NaN,2026/02/22 12:53:26 PM,True,shopify
4,46516044366073,aiming-higher-in-olive-tee,4-5,322.0,NaN,2026/02/22 12:53:27 PM,True,shopify
...,...,...,...,...,...,...,...,...
4040,46461443375353,twirl-in-powder-pink-girls-tee,6-7,322.0,NaN,2026/02/22 12:59:37 PM,True,shopify
4041,46461443408121,twirl-in-powder-pink-girls-tee,8-9,322.0,NaN,2026/02/22 12:59:37 PM,True,shopify
4042,46461443440889,twirl-in-powder-pink-girls-tee,10-11,322.0,NaN,2026/02/22 12:59:37 PM,False,shopify
4043,46461443473657,twirl-in-powder-pink-girls-tee,12-13,322.0,NaN,2026/02/22 12:59:37 PM,True,shopify
